# RandomErasing

Visualization for the new `RandomErasing` transform.

- `mode`: shape of the fill — `"zeros"`, `"random_color_uniform"` (one color per image), `"random_color_pixel"` (per-pixel noise)
- `fill`: distribution — `"randn"` (assumes mean/std-normalized inputs) or `"uniform"` (in `[0, max_value]` for float, `[0, 256)` for uint8)

Vectorized across the batch — each image gets its own rectangle / fill in one fused op.

In [ ]:
LITDATA_VAL_PATH = "s3://visionlab-datasets/imagenet1k/pre-processed/s256-l512-jpgbytes-q100-streaming/val/"

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt

from slipstream import (
    SlipstreamDataset,
    SlipstreamLoader,
    DecodeRandomResizedCrop,
    ToTorchImage,
    IMAGENET_MEAN,
    IMAGENET_STD,
)
from slipstream.transforms import Normalize, RandomErasing

dataset = SlipstreamDataset(
    remote_dir=LITDATA_VAL_PATH,
    decode_images=False,
)
print(f"Dataset: {len(dataset):,} samples")

In [ ]:
def load_images(dataset, size=224, batch_size=8):
    """Load a batch of float images in [0, 1], CHW."""
    dec = DecodeRandomResizedCrop(size=size, to_tensor=True, permute=True)
    loader = SlipstreamLoader(
        dataset, batch_size=batch_size, shuffle=False,
        pipelines={'image': [dec]},
        exclude_fields=['path'], verbose=False,
    )
    batch = next(iter(loader))
    loader.shutdown()
    return batch['image'].float() / 255.0


def show_grid(rows, row_labels=None, suptitle=None,
              mean=None, std=None, max_cols=8):
    """Show one or more rows of [B, C, H, W] tensors. Optionally denormalize."""
    if isinstance(rows, torch.Tensor):
        rows = [rows]
    n_rows = len(rows)
    n_cols = min(max_cols, rows[0].shape[0])
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(2.0 * n_cols, 2.2 * n_rows))
    if n_rows == 1:
        axes = axes[np.newaxis, :]
    for r, batch in enumerate(rows):
        for c in range(n_cols):
            img = batch[c].detach().clone().float()
            if mean is not None and std is not None:
                m = torch.tensor(mean).view(-1, 1, 1)
                s = torch.tensor(std).view(-1, 1, 1)
                img = img * s + m
            img = img.permute(1, 2, 0).clamp(0, 1).cpu().numpy()
            axes[r, c].imshow(img)
            axes[r, c].axis('off')
        if row_labels:
            axes[r, 0].set_title(row_labels[r], fontsize=10, loc='left', x=-0.05, y=0.4)
    if suptitle:
        fig.suptitle(suptitle, fontsize=13, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()


images = load_images(dataset, size=224, batch_size=8)
print(f"images: {tuple(images.shape)}, dtype={images.dtype}, range=[{images.min():.3f}, {images.max():.3f}]")

## Three modes side by side

Working in `[0, 1]` float (no normalization), so `fill="uniform"` with `max_value=1.0`.

In [ ]:
kwargs = dict(p=1.0, fill="uniform", max_value=1.0, seed=42, device="cpu")

out_zeros = RandomErasing(mode="zeros",                 **kwargs)(images.clone())
out_uni   = RandomErasing(mode="random_color_uniform",  **kwargs)(images.clone())
out_pix   = RandomErasing(mode="random_color_pixel",    **kwargs)(images.clone())

show_grid(
    [images, out_zeros, out_uni, out_pix],
    row_labels=["original", "zeros", "random_color_uniform", "random_color_pixel"],
    suptitle="RandomErasing modes (fill='uniform', float in [0,1])",
)

## Per-sample randomness at p=0.5

Each image rolls its own Bernoulli — some get erased, some don't, all with different rectangles. No Python loop over the batch.

In [ ]:
t = RandomErasing(p=0.5, mode="random_color_pixel", fill="uniform", seed=7, device="cpu")
out = t(images.clone())
params = t.last_params()
labels = ["erased" if bool(params["do"][i]) else "untouched" for i in range(images.shape[0])]
print("per-sample:", labels)

show_grid([images, out], row_labels=["original", "p=0.5"], suptitle="Per-sample erasing")

## fill="randn" on normalized inputs (timm-parity)

When applying erasing **after** mean/std normalization, `fill="randn"` produces noise that statistically resembles normalized image data. We denormalize for visualization.

In [ ]:
norm = Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD, device="cpu")
images_norm = norm(images.clone())
print(f"normalized range: [{images_norm.min():.2f}, {images_norm.max():.2f}]")

erase = RandomErasing(p=1.0, mode="random_color_pixel", fill="randn", seed=42, device="cpu")
out_norm = erase(images_norm.clone())

# Both rows are in normalized space; show_grid denormalizes both for display.
show_grid(
    [images_norm, out_norm],
    row_labels=["original (normalized)", "erased (randn)"],
    suptitle="fill='randn' applied after Normalize, denormalized for display",
    mean=IMAGENET_MEAN, std=IMAGENET_STD,
)

## Sweep `area_range`

Fraction of image area to erase. timm default is `(0.02, 1/3)`.

In [ ]:
rows = []
labels = []
for ar in [(0.02, 0.05), (0.05, 0.15), (0.15, 1/3), (1/3, 0.6)]:
    t = RandomErasing(p=1.0, area_range=ar, mode="random_color_pixel",
                      fill="uniform", seed=11, device="cpu")
    rows.append(t(images.clone()))
    labels.append(f"area={ar}")
show_grid(rows, row_labels=labels, suptitle="area_range sweep")

## Sweep `aspect_range`

Aspect ratio (h/w) of the erase rectangle, sampled log-uniformly.

In [ ]:
rows = []
labels = []
for ar in [(0.3, 3.3), (1.0, 1.0), (0.1, 0.3), (3.0, 6.0)]:
    t = RandomErasing(p=1.0, aspect_range=ar, area_range=(0.1, 0.2),
                      mode="random_color_pixel", fill="uniform",
                      seed=23, device="cpu")
    rows.append(t(images.clone()))
    labels.append(f"aspect={ar}")
show_grid(rows, row_labels=labels, suptitle="aspect_range sweep")

## Seed reproducibility & replay

In [ ]:
t1 = RandomErasing(p=0.7, seed=2026, fill="uniform", device="cpu")
t2 = RandomErasing(p=0.7, seed=2026, fill="uniform", device="cpu")
out1 = t1(images.clone())
out2 = t2(images.clone())
print(f"same seed → identical output: {torch.equal(out1, out2)}")

out3 = t1.apply_last(images.clone())
print(f"replay matches original call: {torch.equal(out1, out3)}")

## uint8 inputs

`fill="uniform"` with uint8 → `randint(0, 256)`. `fill="randn"` raises (it doesn't make sense for unsigned integer pixels).

In [ ]:
images_u8 = (images * 255).round().to(torch.uint8)
out_u8 = RandomErasing(p=1.0, mode="random_color_pixel", fill="uniform",
                       seed=42, device="cpu")(images_u8.clone())
print(f"input dtype: {images_u8.dtype}, output dtype: {out_u8.dtype}")
print(f"output range: [{out_u8.min()}, {out_u8.max()}]")

show_grid([images_u8.float() / 255, out_u8.float() / 255],
          row_labels=["original (uint8)", "erased (uint8)"],
          suptitle="uint8 path")

## Single-image (3D) input

Same code path — pass a `[C, H, W]` tensor and get a `[C, H, W]` tensor back.

In [ ]:
img = images[0]
print(f"input shape: {tuple(img.shape)}")
out = RandomErasing(p=1.0, mode="random_color_pixel", fill="uniform",
                    seed=42, device="cpu")(img.clone())
print(f"output shape: {tuple(out.shape)}")

fig, axes = plt.subplots(1, 2, figsize=(6, 3))
axes[0].imshow(img.permute(1, 2, 0).clamp(0, 1).numpy()); axes[0].axis('off'); axes[0].set_title('original')
axes[1].imshow(out.permute(1, 2, 0).clamp(0, 1).numpy()); axes[1].axis('off'); axes[1].set_title('erased')
plt.tight_layout(); plt.show()